## Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

    A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
    A function or coroutine to execute.


In [2]:
import os
import langchain
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
langchain.__version__

'1.2.17'

In [3]:
model = init_chat_model("groq:qwen/qwen3-32b")
responce = model.invoke("Why do parrots talk?")
responce

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They\'re birds known for mimicking human speech. But why do they do that? Maybe it\'s related to their social behavior? I\'ve heard that parrots are social animals, so maybe they mimic to communicate with their flock. But in the wild, they don\'t have humans, so talking must be a learned behavior from humans. \n\nI should think about how they learn. Do they associate human words with actions or events? Like if you say "hello" when greeting them, they might learn to say it back. Also, parrots have a certain structure in their vocal cords that allows them to mimic sounds. Maybe their anatomy plays a role. \n\nAnother angle is the concept of social bonding. If they mimic humans, it might help them feel part of the human group, which is similar to how they interact with their flock. Maybe they use mimicry to get attention or to interact with their human caregiver

In [4]:
@tool
def get_weather(location: str) -> str:
    '''Get the weather at location'''
    return f"It is sunny at {location}"

model_with_tools = model.bind_tools([get_weather])

In [6]:
responce = model_with_tools.invoke("What's the weather like in Boston?")
print(responce)

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'wzgrq6c12', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 153, 'total_tokens': 239, 'completion_time': 0.138839803, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.005937139, 'prompt_tokens_details': None, 'queue_time': 0.052642711, 'total_time': 0.144776942}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019df855-79bf-7f71-a

In [9]:
message = [{"role": "user", "content": "What's the weather like in Boston?"}]
ai_message = model_with_tools.invoke(message)

message.append(ai_message)

for tool_call in ai_message.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    message.append(tool_result)

final_responce = model_with_tools.invoke(message)
print(final_responce.content)

The weather in Boston is sunny.


In [10]:
print(message)

[{'role': 'user', 'content': "What's the weather like in Boston?"}, AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call that function with "Boston" as the location. I\'ll make sure the parameters are correctly formatted in JSON. No other tools are available, so this should be the only function call needed.\n', 'tool_calls': [{'id': 'raktj080a', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 153, 'total_tokens': 257, 'completion_time': 0.158680492, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.006738241, 'prompt_tokens_details': None, 'queue_time': 0.053759958, 'total_time': 0.165418733}, 'model_name': 'qwen/qwen3-32b',